In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# 1. Dataset Generation: Network Intrusion Detection
# Features: Protocol, Service, Flag, Auth, Volume, Port
data = {
    'Protocol': ['TCP', 'TCP', 'UDP', 'TCP', 'ICMP', 'TCP', 'TCP', 'UDP', 'TCP', 'ICMP', 'UDP', 'TCP', 'UDP', 'TCP', 'ICMP'],
    'Service': ['HTTP', 'SSH', 'DNS', 'SSH', 'SMTP', 'HTTP', 'SSH', 'HTTP', 'DNS', 'SMTP', 'DNS', 'SSH', 'DNS', 'HTTP', 'SMTP'],
    'Flag': ['SF', 'S0', 'SF', 'S0', 'REJ', 'S0', 'S0', 'SF', 'S0', 'REJ', 'SF', 'S0', 'SF', 'S0', 'REJ'],
    'Auth': ['Success', 'Failure', 'Success', 'Failure', 'Failure', 'Success', 'Failure', 'Failure', 'Failure', 'Success', 'Success', 'Failure', 'Success', 'Success', 'Failure'],
    'Volume': ['Low', 'High', 'Low', 'Medium', 'Low', 'High', 'High', 'Medium', 'High', 'Low', 'Low', 'High', 'Low', 'High', 'Low'],
    'Port': ['Std', 'Std', 'Std', 'Std', 'Non-Std', 'Std', 'Non-Std', 'Non-Std', 'Std', 'Std', 'Std', 'Std', 'Std', 'Std', 'Std'],
    'Intrusion': ['No', 'Yes', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'No', 'No', 'Yes']
}

df = pd.DataFrame(data)

# Preprocessing: Encode categorical variables
encoders = {}
for col in df.columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

X = df.drop('Intrusion', axis=1)
y = df['Intrusion']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def run_model(name, model):
    print(f"\n{'='*10} {name} {'='*10}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

    # Print a simplified structure if it's a single tree
    if hasattr(model, 'tree_'):
        tree_rules = export_text(model, feature_names=list(X.columns))
        print("Decision Logic Extract:")
        print("\n".join(tree_rules.split('\n')[:5])) # Print top 5 lines for brevity

# --- Implementation of 7 Types ---

# 1. ID3 Variant (Iterative Dichotomiser 3)
# Note: Modern sklearn uses an optimized version of CART, but we simulate ID3
# by using 'entropy' (Information Gain).
run_model("1. ID3 (Information Gain)",
          DecisionTreeClassifier(criterion='entropy', max_depth=3))

# 2. C4.5 Variant
# Simulating C4.5 by using Gain Ratio (not directly in sklearn,
# but entropy + limited depth is the standard approximation).
run_model("2. C4.5 (Gain Ratio Approximation)",
          DecisionTreeClassifier(criterion='entropy', splitter='best', min_samples_leaf=2))

# 3. CART (Classification and Regression Trees)
# Uses Gini Impurity as the default splitting metric.
run_model("3. CART (Gini Impurity)",
          DecisionTreeClassifier(criterion='gini'))

# 4. CHAID (Chi-square Automatic Interaction Detector)
# Simulating CHAID-like behavior using shallow trees with high min_samples_split
# (simulating statistical significance requirements).
run_model("4. CHAID (Statistical Interaction Sim)",
          DecisionTreeClassifier(min_samples_split=4, max_depth=2))

# 5. MARS (Multivariate Adaptive Regression Splines)
# In classification, this often manifests as a Decision Tree with specific pruning.
run_model("5. MARS (Pruned Logic)",
          DecisionTreeClassifier(ccp_alpha=0.01))

# 6. Random Forest
# Building an ensemble of Bagged Decision Trees.
run_model("6. Random Forest (Ensemble)",
          RandomForestClassifier(n_estimators=100, random_state=42))

# 7. Gradient Boosted Decision Trees (GBDT)
# Sequential error correction via trees.
run_model("7. Gradient Boosted Trees",
          GradientBoostingClassifier(n_estimators=50, learning_rate=0.1))

print("\nProcessing complete. All 7 models evaluated on Cybersecurity data.")


========== 1. ID3 (Information Gain) ==========
Accuracy: 1.00
Decision Logic Extract:
|--- Auth <= 0.50
|   |--- Flag <= 1.50
|   |   |--- class: 1
|   |--- Flag >  1.50
|   |   |--- class: 0

========== 2. C4.5 (Gain Ratio Approximation) ==========
Accuracy: 0.67
Decision Logic Extract:
|--- Service <= 1.50
|   |--- Auth <= 0.50
|   |   |--- class: 0
|   |--- Auth >  0.50
|   |   |--- class: 0

========== 3. CART (Gini Impurity) ==========
Accuracy: 0.67
Decision Logic Extract:
|--- Service <= 1.50
|   |--- Auth <= 0.50
|   |   |--- Protocol <= 1.50
|   |   |   |--- class: 1
|   |   |--- Protocol >  1.50

========== 4. CHAID (Statistical Interaction Sim) ==========
Accuracy: 0.67
Decision Logic Extract:
|--- Service <= 1.50
|   |--- Auth <= 0.50
|   |   |--- class: 0
|   |--- Auth >  0.50
|   |   |--- class: 0

========== 5. MARS (Pruned Logic) ==========
Accuracy: 0.67
Decision Logic Extract:
|--- Service <= 1.50
|   |--- Auth <= 0.50
|   |   |--- Port <= 0.50
|   |   |   |--- clas